In [1]:
import random
from datetime import datetime, timedelta

# Create a large sample dataset of vehicle telemetry
file_name = "raw_fleet_telemetry.csv"
start_time = datetime.now()

fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE", "NONE"]

print("⚡ Manufacturing 10,000 data rows...")

with open(file_name, "w") as f:
    # Write the CSV header (column names)
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    
    for i in range(10000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(000000, 999999)}"
        rpm = random.randint(1500, 3500) if random.random() > 0.05 else 0  # simulate occasional engine off
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        
        # Write a single comma-separated row
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")
                # ,{v_id},{rpm},{temp},{code}\n")

print(f"📝 Step 1 Complete: Created '{file_name}' with 10,000 diagnostic data rows!") 

⚡ Manufacturing 10,000 data rows...
📝 Step 1 Complete: Created 'raw_fleet_telemetry.csv' with 10,000 diagnostic data rows!


In [3]:
# Step 2: The Memory-Safe Streamer (ItemReader)
def telemetry_streamer(file_path):
    """A generator function that streams rows one-by-one without overloading RAM."""
    with open(file_path, "r") as f:
        # Skip the header row so we don't treat column names as data
        header = f.readline()
        
        for line in f:
            # Strip off the newline character and split the comma-separated values
            row = line.strip().split(",")
            
            # Pack the raw list into a clean dictionary structure
            data_entry = {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }
            
            # Yield pauses the function here and streams this single row outward
            yield data_entry

print("⚙️ Step 2 Complete: Memory-safe intake valve logic defined!")

⚙️ Step 2 Complete: Memory-safe intake valve logic defined!


In [4]:
# Test-firing the intake valve
stream = telemetry_streamer("raw_fleet_telemetry.csv")

print("🔍 Inspecting the first 3 streamed items on the conveyor belt:\n")
print(next(stream))
print(next(stream))
print(next(stream))

🔍 Inspecting the first 3 streamed items on the conveyor belt:

{'timestamp': '2026-06-18 17:22:32.764561', 'vehicle_id': 'VIN-100', 'engine_rpm': 2709, 'coolant_temp_c': 86, 'fault_code': 'P0300'}
{'timestamp': '2026-06-18 17:22:33.764561', 'vehicle_id': 'VIN-105', 'engine_rpm': 2238, 'coolant_temp_c': 100, 'fault_code': 'NONE'}
{'timestamp': '2026-06-18 17:22:34.764561', 'vehicle_id': 'VIN-105', 'engine_rpm': 0, 'coolant_temp_c': 95, 'fault_code': 'P0300'}


In [6]:
# Step 3: The Core Filter (ItemProcessor)
def telemetry_processor(data_entry):
    """Inspects a single streamed data entry and flags it if a trouble code exists."""
    # If there is no fault code, we return None (skip the record)
    if data_entry["fault_code"] == "NONE":
        return None
    
    # If a code is found, we enrich the data entry with an alert level
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    
    return processed_entry

print("⚙️ Step 3 Complete: Engine core processing logic defined!")

⚙️ Step 3 Complete: Engine core processing logic defined!


In [7]:
# Create a small test list based on your actual sample data
sample_batch = [
    {'timestamp': '2026-06-06 10:51:51.981650', 'vehicle_id': 'VIN-964243', 'engine_rpm': 2138, 'coolant_temp_c': 104, 'fault_code': 'NONE'},
    {'timestamp': '2026-06-06 10:51:52.981650', 'vehicle_id': 'VIN-698090', 'engine_rpm': 2989, 'coolant_temp_c': 103, 'fault_code': 'P0171'}
]

print("🔍 Simulating filter runs:")
for item in sample_batch:
    result = telemetry_processor(item)
    print(f"\nInput Fault Code: {item['fault_code']} -> Processor Output: {result}")

🔍 Simulating filter runs:

Input Fault Code: NONE -> Processor Output: None

Input Fault Code: P0171 -> Processor Output: {'timestamp': '2026-06-06 10:51:52.981650', 'vehicle_id': 'VIN-698090', 'engine_rpm': 2989, 'coolant_temp_c': 103, 'fault_code': 'P0171', 'alert_status': '⚠️ CRITICAL DEVIATION DETECTED', 'processed_at': '2026-06-18 17:24:07.707541'}


In [8]:
# Step 4: The Bulk Persistent Writer (ItemWriter)
def telemetry_writer(output_file_path, processed_entry):
    """Appends a single enriched record into our permanent storage file."""
    # We only write to disk if there is actual data passed down from the processor
    if processed_entry is not None:
        with open(output_file_path, "a") as f:
            # Create a clean line string from the dictionary values
            line = (
                f"{processed_entry['timestamp']},"
                f"{processed_entry['vehicle_id']},"
                f"{processed_entry['engine_rpm']},"
                f"{processed_entry['coolant_temp_c']},"
                f"{processed_entry['fault_code']},"
                f"{processed_entry['alert_status']},"
                f"{processed_entry['processed_at']}\n"
            )
            f.write(line)

print("⚙️ Step 4 Complete: Outtake writer logic defined!")

⚙️ Step 4 Complete: Outtake writer logic defined!


In [ ]:
En las practicas recientes hemos mejorado las gestiones dinamicas de rutas usando pathlib en vez de os.path para que las carpetas funcionen tanto local como en servidores, ejemplo: INPUT_DIR = BASE_DIR / "data_input"
OUTPUT_DIR = BASE_DIR / "logs", y hemos hecho funciones para iniciar entorno con try: y con except, garantizando la existencia de las carpetas algo asi: NPUT_DIR.mkdir(parents=True, exist_ok=True)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True), mi pregunta, seria bueno modernizar el telemetr_streame, processor and writer con esta technica mencionada o no es necesario, lo digo por que estoy aprendiendo a estar seguro que las carpetas y archivos van a ser encontradas tanto local como en la nube dependiendo el caso y no se si en el caso de las funciones de telemetry que hemos practicado esten al tanto de esta situacion, aqui te muestro un codigo fuente de telemetry: # 1. Empaquetamos el script limpio en la variable pipeline_code
batch_script = """import random
from datetime import datetime
import os

def telemetry_streamer(file_path):
    with open(file_path, "r") as f:
        header = f.readline()
        for line in f:
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def telemetry_writer(output_file_path, processed_entry):
    if processed_entry is not None:
        with open(output_file_path, "a") as f:
            line = f"{processed_entry['timestamp']},{processed_entry['vehicle_id']},{processed_entry['engine_rpm']},{processed_entry['coolant_temp_c']},{processed_entry['fault_code']},{processed_entry['alert_status']},{processed_entry['processed_at']}\\n"
            f.write(line)

if __name__ == "__main__":
    input_source = "/app/data/raw_fleet_telemetry.csv"
    output_destination = "/app/data/clean_critical_incidents.csv"
    
    if not os.path.exists(input_source):
        print(f"❌ Error: Input file {input_source} not found!")
        exit(1)
        
    with open(output_destination, "w") as f:
        f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\\n")
        
    print("🚀 Cloud Batch Engine running...")
    stream = telemetry_streamer(input_source)
    count = 0
    for raw in stream:
        enriched = telemetry_processor(raw)
        if enriched:
            telemetry_writer(output_destination, enriched)
            count += 1
            
    print(f"🏁 Batch complete! Wrote {count} incidents to {output_destination}")
"""

with open("batch_worker.py", "w") as f:
    f.write(batch_script)

print("💾 batch_worker.py has been written successfully!") 


In [9]:
# The Master Enterprise Batch Job Orchestrator
input_source = "raw_fleet_telemetry.csv"
output_destination = "clean_critical_incidents.csv"

# Initialize a fresh output file with a new header row
with open(output_destination, "w") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n")

print(f"🚀 Firing up the background engine loop...")

# 1. Start the intake valve streamer (ItemReader)
data_stream = telemetry_streamer(input_source)

processed_count = 0

# Loop through the infinite generator line memory-safely
for raw_record in data_stream:
    # 2. Pass record through the filter core (ItemProcessor)
    enriched_record = telemetry_processor(raw_record)
    
    # 3. Hand off the result to the file system (ItemWriter)
    if enriched_record:
        telemetry_writer(output_destination, enriched_record)
        processed_count += 1

print(f"🏁 Batch Job Complete! Processed 10,000 lines.")
print(f"📂 Caught and wrote {processed_count} critical incidents to '{output_destination}'!")

🚀 Firing up the background engine loop...
🏁 Batch Job Complete! Processed 10,000 lines.
📂 Caught and wrote 4347 critical incidents to 'clean_critical_incidents.csv'!


In [ ]:
4245

In [10]:
# Step 5: Stamping the unified script onto the disk
batch_script = """import random
from datetime import datetime
import os

def telemetry_streamer(file_path):
    with open(file_path, "r") as f:
        header = f.readline()
        for line in f:
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def telemetry_writer(output_file_path, processed_entry):
    if processed_entry is not None:
        with open(output_file_path, "a") as f:
            line = f"{processed_entry['timestamp']},{processed_entry['vehicle_id']},{processed_entry['engine_rpm']},{processed_entry['coolant_temp_c']},{processed_entry['fault_code']},{processed_entry['alert_status']},{processed_entry['processed_at']}\\n"
            f.write(line)

if __name__ == "__main__":
    input_source = "/app/data/raw_fleet_telemetry.csv"
    output_destination = "/app/data/clean_critical_incidents.csv"
    
    if not os.path.exists(input_source):
        print(f"❌ Error: Input file {input_source} not found!")
        exit(1)
        
    with open(output_destination, "w") as f:
        f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\\n")
        
    print("🚀 Cloud Batch Engine running...")
    stream = telemetry_streamer(input_source)
    count = 0
    for raw in stream:
        enriched = telemetry_processor(raw)
        if enriched:
            telemetry_writer(output_destination, enriched)
            count += 1
            
    print(f"🏁 Batch complete! Wrote {count} incidents to {output_destination}")
"""

with open("batch_worker.py", "w") as f:
    f.write(batch_script)

print("💾 batch_worker.py has been written successfully!")

💾 batch_worker.py has been written successfully!


In [11]:
# Step 6: Writing the lean background worker Dockerfile
worker_blueprint = """FROM python:3.11-slim
WORKDIR /app
COPY batch_worker.py .
CMD ["python", "batch_worker.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(worker_blueprint)

print("🐳 Lean background worker Dockerfile successfully written!")

🐳 Lean background worker Dockerfile successfully written!


In [12]:
!docker build -t fleet-batch-worker .

Cannot connect to the Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?


In [13]:
import os
import shutil

# 1. Create a clean, dedicated data directory on your physical Mac
local_data_dir = os.path.join(os.getcwd(), "fleet_data")
if not os.path.exists(local_data_dir):
    os.makedirs(local_data_dir)
    print(f"📁 Created local folder: {local_data_dir}")

# 2. Move our raw data pile into that dedicated folder
src_file = "raw_fleet_telemetry.csv"
dest_file = os.path.join(local_data_dir, "raw_fleet_telemetry.csv")

if os.path.exists(src_file):
    shutil.move(src_file, dest_file)
    print(f"📦 Moved raw telemetry file into: {dest_file}")

print("\n⚡ Setup complete. Ready to run the Docker command below!")

📦 Moved raw telemetry file into: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fourth_container/flat_code_entetrprise/fleet_data/raw_fleet_telemetry.csv

⚡ Setup complete. Ready to run the Docker command below!


In [14]:
!docker run --rm -v $(pwd)/fleet_data:/app/data fleet-batch-worker

docker: Cannot connect to the Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?.
See 'docker run --help'.


In [15]:
!docker run --rm -v "$(pwd)/fleet_data:/app/data" fleet-batch-worker

docker: Cannot connect to the Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?.
See 'docker run --help'.


In [ ]:
# ==========================================
# 1. DEFINICIÓN DE ENTORNOS CON PATHLIB
# ==========================================
# Cambiamos Path(__file__) por Path.cwd() para entornos interactivos
BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "data" / "input"
OUTPUT_DIR = BASE_DIR / "data" / "output"

# Garantizar de manera defensiva que existan las carpetas
try:
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Entorno listo en el directorio de trabajo: {BASE_DIR}")
except Exception as e:
    print(f"❌ Error al inicializar el entorno de carpetas: {e}")
    sys.exit(1)

In [5]:
import random
from datetime import datetime, timedelta
from pathlib import Path
import sys

# ==========================================
# 1. DEFINICIÓN DE ENTORNOS CON PATHLIB
# ==========================================
BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "data" / "input"
OUTPUT_DIR = BASE_DIR / "data" / "output"

# Garantizar de manera defensiva que existan las carpetas
try:
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Entorno listo en el directorio de trabajo: {BASE_DIR}")
except Exception as e:
    print(f"❌ Error al inicializar el entorno de carpetas: {e}")
    sys.exit(1)

# Definir las rutas exactas de los archivos usando objetos Path
input_file_path = INPUT_DIR / "raw_fleet_telemetry.csv"
output_file_path = OUTPUT_DIR / "clean_critical_incidents.csv"

# ==========================================
# 2. GENERADOR DE DATOS (Tus 10,000 filas)
# ==========================================
start_time = datetime.now()
fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE", "NONE"]

print("⚡ Manufacturing 10,000 data rows into input directory...")

with open(input_file_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    
    for i in range(10000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(000000, 999999)}"
        rpm = random.randint(1500, 3500) if random.random() > 0.05 else 0  
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"📝 Step 1 Complete: Created '{input_file_path.name}' with 10,000 rows!")

# ==========================================
# 3. PIPELINE DE PROCESAMIENTO (Batch Worker Moderno)
# ==========================================
def telemetry_streamer(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        header = f.readline()  # Omitir encabezado
        for line in f:
            if not line.strip():
                continue
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def telemetry_writer(output_path: Path, processed_entry):
    with open(output_path, "a", encoding="utf-8") as f:
        line = (
            f"{processed_entry['timestamp']},"
            f"{processed_entry['vehicle_id']},"
            f"{processed_entry['engine_rpm']},"
            f"{processed_entry['coolant_temp_c']},"
            f"{processed_entry['fault_code']},"
            f"{processed_entry['alert_status']},"
            f"{processed_entry['processed_at']}\n"
        )
        f.write(line)

# ==========================================
# 4. EJECUCIÓN DEL PROCESO
# ==========================================
if input_file_path.exists():
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n")
        
    print("\n🚀 Cloud Batch Engine running with Pathlib (Processing 10k rows)...")
    
    stream = telemetry_streamer(input_file_path)
    count = 0
    
    for raw in stream:
        enriched = telemetry_processor(raw)
        if enriched:
            telemetry_writer(output_file_path, enriched)
            count += 1
            
    print(f"🏁 Batch complete! Wrote {count} critical diagnostic incidents to {output_file_path.relative_to(BASE_DIR)}")
else:
    print(f"❌ Error: El archivo generado no se encuentra en {input_file_path}")

📁 Entorno listo en el directorio de trabajo: /Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/fourth_container/flat_code_entetrprise
⚡ Manufacturing 10,000 data rows into input directory...
📝 Step 1 Complete: Created 'raw_fleet_telemetry.csv' with 10,000 rows!

🚀 Cloud Batch Engine running with Pathlib (Processing 10k rows)...
🏁 Batch complete! Wrote 4245 critical diagnostic incidents to data/output/clean_critical_incidents.csv


In [3]:
# Step 5: Stamping the unified script onto the disk
batch_script = """import random
from datetime import datetime, timedelta
from pathlib import Path
import sys

# ==========================================
# 1. DEFINICIÓN DE ENTORNOS CON PATHLIB
# ==========================================
BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "data" / "input"
OUTPUT_DIR = BASE_DIR / "data" / "output"

# Garantizar de manera defensiva que existan las carpetas
try:
    INPUT_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📁 Entorno listo en el directorio de trabajo: {BASE_DIR}")
except Exception as e:
    print(f"❌ Error al inicializar el entorno de carpetas: {e}")
    sys.exit(1)

# Definir las rutas exactas de los archivos usando objetos Path
input_file_path = INPUT_DIR / "raw_fleet_telemetry.csv"
output_file_path = OUTPUT_DIR / "clean_critical_incidents.csv"

# ==========================================
# 2. GENERADOR DE DATOS (Tus 10,000 filas)
# ==========================================
start_time = datetime.now()
fault_codes = ["P0171", "P0300", "P0420", "NONE", "NONE", "NONE", "NONE"]

print("⚡ Manufacturing 10,000 data rows into input directory...")

with open(input_file_path, "w", encoding="utf-8") as f:
    f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code\n")
    
    for i in range(10000):
        timestamp = start_time + timedelta(seconds=i)
        v_id = f"VIN-{random.randint(000000, 999999)}"
        rpm = random.randint(1500, 3500) if random.random() > 0.05 else 0  
        temp = random.randint(85, 105)
        code = random.choice(fault_codes)
        
        f.write(f"{timestamp},{v_id},{rpm},{temp},{code}\n")

print(f"📝 Step 1 Complete: Created '{input_file_path.name}' with 10,000 rows!")

# ==========================================
# 3. PIPELINE DE PROCESAMIENTO (Batch Worker Moderno)
# ==========================================
def telemetry_streamer(file_path: Path):
    with open(file_path, "r", encoding="utf-8") as f:
        header = f.readline()  # Omitir encabezado
        for line in f:
            if not line.strip():
                continue
            row = line.strip().split(",")
            yield {
                "timestamp": row[0],
                "vehicle_id": row[1],
                "engine_rpm": int(row[2]),
                "coolant_temp_c": int(row[3]),
                "fault_code": row[4]
            }

def telemetry_processor(data_entry):
    if data_entry["fault_code"] == "NONE":
        return None
    
    processed_entry = data_entry.copy()
    processed_entry["alert_status"] = "⚠️ CRITICAL DEVIATION DETECTED"
    processed_entry["processed_at"] = str(datetime.now())
    return processed_entry

def telemetry_writer(output_path: Path, processed_entry):
    with open(output_path, "a", encoding="utf-8") as f:
        line = (
            f"{processed_entry['timestamp']},"
            f"{processed_entry['vehicle_id']},"
            f"{processed_entry['engine_rpm']},"
            f"{processed_entry['coolant_temp_c']},"
            f"{processed_entry['fault_code']},"
            f"{processed_entry['alert_status']},"
            f"{processed_entry['processed_at']}\n"
        )
        f.write(line)

# ==========================================
# 4. EJECUCIÓN DEL PROCESO
# ==========================================
if input_file_path.exists():
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write("timestamp,vehicle_id,engine_rpm,coolant_temp_c,fault_code,alert_status,processed_at\n")
        
    print("\n🚀 Cloud Batch Engine running with Pathlib (Processing 10k rows)...")
    
    stream = telemetry_streamer(input_file_path)
    count = 0
    
    for raw in stream:
        enriched = telemetry_processor(raw)
        if enriched:
            telemetry_writer(output_file_path, enriched)
            count += 1
            
    print(f"🏁 Batch complete! Wrote {count} critical diagnostic incidents to {output_file_path.relative_to(BASE_DIR)}")
else:
    print(f"❌ Error: El archivo generado no se encuentra en {input_file_path}")"""

with open("pipeline.py", "w") as f:
    f.write(batch_script)

print("💾 pipeline.py.py has been written successfully!")

💾 pipeline.py.py has been written successfully!


In [4]:
# Step 6: Writing the lean background worker Dockerfile
worker_blueprint = """FROM python:3.11-slim
WORKDIR /app
COPY pipeline.py .
CMD ["python", "pipeline.py.py"]
"""

with open("Dockerfile", "w") as f:
    f.write(worker_blueprint)

print("🐳 Lean background worker Dockerfile successfully written!")

🐳 Lean background worker Dockerfile successfully written!


In [ ]:
user1s-MacBook-Pro:enterprise admin$ docker run --rm -v "$(pwd)/data:/app/data" telemetry-processor:v1


In [ ]:
 user1s-MacBook-Pro:enterprise admin$ docker run --rm -v "$(pwd)/data:/app/data" telemetry-processor:v1
📁 Entorno listo en el directorio de trabajo: /app
⚡ Manufacturing 10,000 data rows into input directory...
📝 Step 1 Complete: Created 'raw_fleet_telemetry.csv' with 10,000 rows!

🚀 Cloud Batch Engine running with Pathlib (Processing 10k rows)...